# 01 — Data Generation
Generates a realistic 6-month synthetic electronics e-commerce dataset and loads it into SQLite.

**Tables:** `users` · `products` · `sessions` · `events` · `orders` · `order_items`  
**Period:** 2024-01-01 → 2024-06-30

In [2]:
# Cell 1 — Imports & Paths
import sqlite3
import random
import warnings
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')
fake = Faker()
random.seed(42)
np.random.seed(42)

BASE_DIR   = Path('..').resolve()
DB_PATH    = BASE_DIR / 'data' / 'raw' / 'ecommerce.db'
SCHEMA_SQL = BASE_DIR / 'sql' / 'schema.sql'
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Imports OK')
print(f'DB will be saved to: {DB_PATH}')

Imports OK
DB will be saved to: C:\Users\diyag\Desktop\projects\ecommerce\ecommerce-funnel-analysis\data\raw\ecommerce.db


In [3]:
# Cell 2 — Config & Constants
N_USERS    = 5_000
N_PRODUCTS = 120
START_DATE = datetime(2024, 1, 1)
END_DATE   = datetime(2024, 6, 30)
DATE_RANGE = (END_DATE - START_DATE).days

COUNTRIES       = ['US','IN','UK','CA','AU','DE','FR','SG','AE','BR']
COUNTRY_W       = [0.30,0.18,0.10,0.08,0.06,0.07,0.05,0.04,0.04,0.08]
DEVICES         = ['desktop','mobile','tablet']
DEVICE_W        = [0.40, 0.50, 0.10]
CHANNELS        = ['organic_search','paid_search','social_media','email','direct','referral']
CHANNEL_W       = [0.28, 0.22, 0.18, 0.12, 0.12, 0.08]
PAYMENT_METHODS = ['credit_card','debit_card','paypal','upi','buy_now_pay_later']
AGE_GROUPS      = ['18-24','25-34','35-44','45-54','55+']
ORDER_STATUSES  = ['completed','returned','cancelled']
ORDER_STATUS_W  = [0.88, 0.07, 0.05]

FUNNEL = {
    'product_view_rate':   0.72,
    'add_to_cart_rate':    0.38,
    'checkout_start_rate': 0.55,
    'purchase_rate':       0.68,
}

CHANNEL_CONV_MULT = {
    'email': 1.25, 'paid_search': 1.10, 'direct': 1.05,
    'organic_search': 1.00, 'referral': 0.95, 'social_media': 0.85
}

LANDING_PAGES = ['/home','/deals','/new-arrivals','/category/smartphones',
                 '/category/laptops','/category/headphones','/sale']

print('Config loaded.')

Config loaded.


In [4]:
# Cell 3 — Generate Products
CATALOG = [
    ('Smartphones','Samsung','Galaxy S24 Ultra',1199,0.55),
    ('Smartphones','Apple','iPhone 15 Pro',1099,0.52),
    ('Smartphones','OnePlus','OnePlus 12',799,0.50),
    ('Smartphones','Google','Pixel 8 Pro',999,0.53),
    ('Smartphones','Xiaomi','Redmi Note 13 Pro',349,0.48),
    ('Smartphones','Samsung','Galaxy A54',449,0.50),
    ('Smartphones','Apple','iPhone 14',699,0.52),
    ('Laptops','Apple','MacBook Pro 14-inch',1999,0.58),
    ('Laptops','Dell','XPS 15',1799,0.55),
    ('Laptops','HP','Spectre x360',1499,0.54),
    ('Laptops','Lenovo','ThinkPad X1 Carbon',1649,0.56),
    ('Laptops','ASUS','ROG Zephyrus G14',1399,0.54),
    ('Laptops','Acer','Swift 3',649,0.50),
    ('Laptops','Microsoft','Surface Pro 9',1299,0.55),
    ('Tablets','Apple','iPad Pro 12.9',1099,0.54),
    ('Tablets','Samsung','Galaxy Tab S9',799,0.52),
    ('Tablets','Amazon','Fire HD 10',149,0.45),
    ('Tablets','Lenovo','Tab P12 Pro',599,0.50),
    ('Headphones','Sony','WH-1000XM5',349,0.45),
    ('Headphones','Bose','QuietComfort 45',329,0.44),
    ('Headphones','Apple','AirPods Pro 2',249,0.46),
    ('Headphones','Jabra','Evolve2 75',449,0.47),
    ('Headphones','Sennheiser','HD 660S2',499,0.48),
    ('Headphones','Samsung','Galaxy Buds2 Pro',199,0.44),
    ('Smart Watches','Apple','Apple Watch Series 9',399,0.50),
    ('Smart Watches','Samsung','Galaxy Watch 6 Classic',349,0.48),
    ('Smart Watches','Garmin','Fenix 7X',799,0.52),
    ('Smart Watches','Fitbit','Sense 2',249,0.46),
    ('Cameras','Sony','Alpha 7 IV',2499,0.60),
    ('Cameras','Canon','EOS R6 Mark II',2499,0.60),
    ('Cameras','Nikon','Z6 III',1999,0.58),
    ('Cameras','Fujifilm','X-T5',1699,0.57),
    ('Cameras','GoPro','Hero 12 Black',399,0.50),
    ('TVs','Samsung','QLED 65-inch 4K',1299,0.54),
    ('TVs','LG','OLED C3 65-inch',1799,0.56),
    ('TVs','Sony','Bravia XR 55-inch',1199,0.54),
    ('TVs','TCL','75-inch QLED 4K',699,0.50),
    ('Gaming','Sony','PlayStation 5',499,0.55),
    ('Gaming','Microsoft','Xbox Series X',499,0.55),
    ('Gaming','Nintendo','Switch OLED',349,0.52),
    ('Gaming','Valve','Steam Deck 512GB',649,0.54),
    ('Accessories','Anker','PowerBank 26800mAh',59,0.40),
    ('Accessories','Belkin','MagSafe Charger',39,0.38),
    ('Accessories','Logitech','MX Master 3S Mouse',99,0.42),
    ('Accessories','Apple','Magic Keyboard',129,0.44),
    ('Accessories','Samsung','65W USB-C Charger',49,0.38),
]

products_rows = []
for pid in range(1, N_PRODUCTS + 1):
    subcat, brand, name, base_price, cost_pct = CATALOG[(pid - 1) % len(CATALOG)]
    v      = 1.0 if pid <= len(CATALOG) else round(random.uniform(0.88, 1.12), 2)
    price  = round(base_price * v, 2)
    suffix = f' (v{pid // len(CATALOG) + 1})' if pid > len(CATALOG) else ''
    products_rows.append((
        pid, name + suffix, 'Electronics', subcat, brand,
        price, round(price * cost_pct, 2), random.randint(50, 500)
    ))

df_products = pd.DataFrame(products_rows,
    columns=['product_id','product_name','category','subcategory',
             'brand','price','cost','stock_quantity'])

print(f'Products generated: {len(df_products)}')
print(df_products.groupby('subcategory').size().sort_values(ascending=False).to_string())

Products generated: 120
subcategory
Laptops          21
Smartphones      21
Headphones       18
Tablets          12
Smart Watches    12
Accessories      10
Cameras          10
Gaming            8
TVs               8


In [5]:
# Cell 4 — Generate Users
users_rows = []
for uid in range(1, N_USERS + 1):
    offset      = random.randint(-180, DATE_RANGE)
    signup_date = (START_DATE + timedelta(days=max(0, offset))).strftime('%Y-%m-%d')
    users_rows.append((
        uid,
        signup_date,
        random.choices(COUNTRIES, COUNTRY_W)[0],
        random.choices(DEVICES,   DEVICE_W)[0],
        random.choices(CHANNELS,  CHANNEL_W)[0],
        random.choice(AGE_GROUPS),
        fake.email(),
    ))

df_users = pd.DataFrame(users_rows,
    columns=['user_id','signup_date','country','device_type',
             'acquisition_channel','age_group','email'])

print(f'Users generated: {len(df_users)}')
print(df_users['acquisition_channel'].value_counts().to_string())

Users generated: 5000
acquisition_channel
organic_search    1416
paid_search       1107
social_media       854
direct             608
email              602
referral           413


In [6]:
# Cell 5 — Generate Sessions, Events, Orders, Order Items
sessions_rows    = []
events_rows      = []
orders_rows      = []
order_items_rows = []

session_id = 1
event_id   = 1
order_id   = 1
item_id    = 1

# Hour-of-day weights (realistic traffic pattern)
HOUR_WEIGHTS = np.array([1,1,1,1,1,1,2,3,4,5,6,6,5,5,5,5,6,6,5,5,4,4,3,2], dtype=float)
HOUR_WEIGHTS /= HOUR_WEIGHTS.sum()   # normalise so sum == 1.0 exactly

# Each user gets a random number of sessions
session_counts = np.random.negative_binomial(3, 0.3, N_USERS).clip(1, 20)

# Build product price lookup for fast access
price_lookup = df_products.set_index('product_id')['price'].to_dict()
product_ids  = df_products['product_id'].tolist()

for uid in range(1, N_USERS + 1):
    user         = df_users.iloc[uid - 1]
    n_sessions   = session_counts[uid - 1]
    channel_mult = CHANNEL_CONV_MULT.get(user['acquisition_channel'], 1.0)

    for _ in range(n_sessions):
        sess_date = START_DATE + timedelta(days=random.randint(0, DATE_RANGE))
        sess_hour = int(np.random.choice(range(24), p=HOUR_WEIGHTS))
        device    = random.choices(DEVICES,  DEVICE_W)[0]
        channel   = random.choices(CHANNELS, CHANNEL_W)[0]
        landing   = random.choice(LANDING_PAGES)

        sessions_rows.append((
            session_id, uid,
            sess_date.strftime('%Y-%m-%d'), sess_hour,
            device, channel, landing
        ))

        ts = sess_date.replace(hour=sess_hour, minute=random.randint(0, 59))

        # — page_view (always happens) —
        events_rows.append((
            event_id, session_id, uid, 'page_view',
            None, ts.strftime('%Y-%m-%d %H:%M:%S'), landing
        ))
        event_id += 1
        ts += timedelta(seconds=random.randint(5, 60))

        # — product_view —
        if random.random() < FUNNEL['product_view_rate']:
            viewed_pid = random.choice(product_ids)
            events_rows.append((
                event_id, session_id, uid, 'product_view',
                viewed_pid, ts.strftime('%Y-%m-%d %H:%M:%S'), f'/product/{viewed_pid}'
            ))
            event_id += 1
            ts += timedelta(seconds=random.randint(20, 180))

            # — add_to_cart —
            if random.random() < FUNNEL['add_to_cart_rate']:
                cart_pids = [viewed_pid]
                if random.random() < 0.25:           # 25% chance of adding a 2nd item
                    cart_pids.append(random.choice(product_ids))

                for cpid in cart_pids:
                    events_rows.append((
                        event_id, session_id, uid, 'add_to_cart',
                        cpid, ts.strftime('%Y-%m-%d %H:%M:%S'), '/cart'
                    ))
                    event_id += 1
                ts += timedelta(seconds=random.randint(10, 90))

                # — checkout_start —
                if random.random() < FUNNEL['checkout_start_rate']:
                    events_rows.append((
                        event_id, session_id, uid, 'checkout_start',
                        None, ts.strftime('%Y-%m-%d %H:%M:%S'), '/checkout'
                    ))
                    event_id += 1
                    ts += timedelta(seconds=random.randint(30, 300))

                    # — purchase —
                    eff_rate = min(FUNNEL['purchase_rate'] * channel_mult, 0.95)
                    if random.random() < eff_rate:
                        events_rows.append((
                            event_id, session_id, uid, 'purchase',
                            None, ts.strftime('%Y-%m-%d %H:%M:%S'), '/order-confirm'
                        ))
                        event_id += 1

                        # Build order
                        discount = round(
                            random.choices([0.0, random.uniform(5, 50)],
                                           weights=[0.55, 0.45])[0], 2)
                        status   = random.choices(ORDER_STATUSES, ORDER_STATUS_W)[0]
                        total    = 0.0

                        for opid in cart_pids:
                            prod_price = price_lookup[opid]
                            qty        = random.choices([1, 2, 3], weights=[0.80, 0.15, 0.05])[0]
                            unit_price = round(prod_price * random.uniform(0.95, 1.05), 2)
                            total     += unit_price * qty
                            order_items_rows.append((
                                item_id, order_id, opid, qty, unit_price
                            ))
                            item_id += 1

                        total = round(max(total - discount, 1.0), 2)
                        orders_rows.append((
                            order_id, session_id, uid,
                            sess_date.strftime('%Y-%m-%d'),
                            total, discount,
                            random.choice(PAYMENT_METHODS),
                            status
                        ))
                        order_id += 1

        session_id += 1

print(f'Sessions    : {len(sessions_rows):>8,}')
print(f'Events      : {len(events_rows):>8,}')
print(f'Orders      : {len(orders_rows):>8,}')
print(f'Order Items : {len(order_items_rows):>8,}')

Sessions    :   34,509
Events      :   79,851
Orders      :    3,548
Order Items :    4,425


In [7]:
# Cell 6 — Build DataFrames & Sanity Check
df_sessions = pd.DataFrame(sessions_rows,
    columns=['session_id','user_id','session_date','session_hour',
             'device_type','channel','landing_page'])

df_events = pd.DataFrame(events_rows,
    columns=['event_id','session_id','user_id','event_type',
             'product_id','timestamp','page'])

df_orders = pd.DataFrame(orders_rows,
    columns=['order_id','session_id','user_id','order_date',
             'total_amount','discount_amount','payment_method','status'])

df_order_items = pd.DataFrame(order_items_rows,
    columns=['item_id','order_id','product_id','quantity','unit_price'])

completed = df_orders[df_orders['status'] == 'completed']

print('=== Event type counts ===')
print(df_events['event_type'].value_counts().to_string())
print()
print(f'Overall session-to-purchase rate : {len(df_orders)/len(df_sessions)*100:.1f}%')
print(f'Completed orders                 : {len(completed):,}')
print(f'Total revenue (completed)        : ${completed["total_amount"].sum():,.0f}')
print(f'Avg order value                  : ${completed["total_amount"].mean():,.2f}')

=== Event type counts ===
event_type
page_view         34509
product_view      24940
add_to_cart       11718
checkout_start     5136
purchase           3548

Overall session-to-purchase rate : 10.3%
Completed orders                 : 3,141
Total revenue (completed)        : $4,040,460
Avg order value                  : $1,286.36


In [8]:
# Cell 7 — Write to SQLite
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)

with open(SCHEMA_SQL, 'r') as f:
    conn.executescript(f.read())

df_users.to_sql('users',             conn, if_exists='append', index=False)
df_products.to_sql('products',       conn, if_exists='append', index=False)
df_sessions.to_sql('sessions',       conn, if_exists='append', index=False)
df_events.to_sql('events',           conn, if_exists='append', index=False)
df_orders.to_sql('orders',           conn, if_exists='append', index=False)
df_order_items.to_sql('order_items', conn, if_exists='append', index=False)

conn.commit()

print('=== Row counts in SQLite ===')
for table in ['users','products','sessions','events','orders','order_items']:
    n = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table:<15} {n:>10,}')

conn.close()
print(f'\nDatabase saved  : {DB_PATH}')
print(f'File size       : {DB_PATH.stat().st_size / 1024 / 1024:.1f} MB')

=== Row counts in SQLite ===
  users                5,000
  products               120
  sessions            34,509
  events              79,851
  orders               3,548
  order_items          4,425

Database saved  : C:\Users\diyag\Desktop\projects\ecommerce\ecommerce-funnel-analysis\data\raw\ecommerce.db
File size       : 11.0 MB


In [9]:
# Cell 8 — Export CSVs for Tableau
PROCESSED = BASE_DIR / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

df_users.to_csv(PROCESSED / 'users.csv',             index=False)
df_products.to_csv(PROCESSED / 'products.csv',       index=False)
df_sessions.to_csv(PROCESSED / 'sessions.csv',       index=False)
df_events.to_csv(PROCESSED / 'events.csv',           index=False)
df_orders.to_csv(PROCESSED / 'orders.csv',           index=False)
df_order_items.to_csv(PROCESSED / 'order_items.csv', index=False)

# Enriched orders table (joins sessions + users) — most useful for Tableau
df_enriched = (
    df_orders
    .merge(df_sessions[['session_id','device_type','channel']], on='session_id', how='left')
    .merge(df_users[['user_id','country','age_group','acquisition_channel']], on='user_id', how='left')
)
df_enriched.to_csv(PROCESSED / 'orders_enriched.csv', index=False)

print('CSVs exported to data/processed/')
for f in sorted(PROCESSED.glob('*.csv')):
    print(f'  {f.name:<30} {f.stat().st_size / 1024:>7.0f} KB')

CSVs exported to data/processed/
  events.csv                        4951 KB
  order_items.csv                     98 KB
  orders.csv                         207 KB
  orders_enriched.csv                341 KB
  products.csv                         8 KB
  sessions.csv                      1938 KB
  users.csv                          327 KB


---
**Done.** SQLite database and CSVs are ready.  
Next: open `02_eda_and_cleaning.ipynb`